## Import essential python packages

In [ ]:
# import spacec first
import spacec as sp
import glob
import numpy as np
import pandas as pd

#import standard packages
import os
import warnings
import matplotlib.pyplot as plt
import scanpy as sc

import anndata as ad
warnings.filterwarnings('ignore')

import fcsparser

from collections import Counter
import matplotlib.backends.backend_pdf
plt.rc('axes', grid=False)  # remove gridlines
sc.settings.set_figure_params(dpi=80, facecolor='white') # set dpi and background color for scanpy figures

## Initialize variables

In [ ]:
# Specify the path to the data
root_path = "~/CODEX_08262025"
data_filename = root_path + "/celldata.xlsx" # where the data is stored

# where you want to store the output
output_dir = root_path + '/Analysis/CN_Corrected_Dormant_Test/' # inset your own path
os.makedirs(output_dir, exist_ok=True)

## Load Ann data

In [ ]:
adata = sc.read_h5ad("%s/Analysis/CN_Corrected/adata_annotated_cn.h5ad" % root_path)
adata

In [ ]:
cond_arr = ["Uninfected" if cond == 3 else "Infected" if cond == 1 else "Prior" for cond in adata.obs['condition']]
final_cond_arr = cond_arr

dormant_codes = ['PD_03', 'PD_04', 'PD_07', 'PD_32', 'PD_33']
uninfected_code = ['PD_08']

for i in range(0, len(cond_arr)):
    if adata.obs.code.values[i] in uninfected_code:
        final_cond_arr[i] = "Uninfected"
    elif adata.obs.code.values[i] in dormant_codes:
        final_cond_arr[i] = "Dormant"
        
Counter(final_cond_arr)

adata.obs['condition_final'] = final_cond_arr

### 3 different cell labels
#### original - from scTyper
#### simple - simplified from original label
#### infalmmatory - by inflammatory state

In [ ]:
targeted = "original"
#targeted = "simple"
#targeted = "inflammatory"

if(targeted == "original"):
    targeted_col = "celltypes"
    
    celltype_palette = ["#F3C300","#875692","#F38400","#A1CAF1","#BE0032",
                          "#C2B280","#848482","#008856","#E68FAC","#0067A5",
                          "#F99379","#604E97","#F6A600","#B3446C","#DCD300",
                          "#882D17","#8DB600","#654522","#E25822","#2B3D26",
                          "#222222","#00C2A0"]
elif(targeted == "simple"):
    targeted_col = "renamed_celltypes"  
    celltype_palette = ["#00202e", "#bc5090", "#ff8531", "#2c4875", "#92ba92", "#c7522a", "#007f4e"]
elif(targeted == "inflammatory"):
    targeted_col = "inflammatory_state"
    #t_meta_data <- t_meta_data %>% filter(inflammatory_state != 'Stromal')
    celltype_palette = ["#00202e", "#bc5090", "#ff8531"]
    

celltype_labels = np.sort(np.unique(adata.obs[targeted_col]))



### Compute CN

In [ ]:
# tune k and n_neighborhoods to obtain the best result
adata = sp.tl.neighborhood_analysis(
    adata, 
    unique_region = "unique_region", 
    cluster_col = targeted_col, 
    X = 'X', Y = 'Y',
    k = 10, # k nearest neighbors
    n_neighborhoods = 20, #number of CNs
    elbow = True)

In [ ]:
# compute for CNs
# tune k and n_neighborhoods to obtain the best result
adata = sp.tl.neighborhood_analysis(
    adata, 
    unique_region = "unique_region", # regions or samples
    cluster_col = targeted_col, # derive clusters from this column
    X = 'X', Y = 'Y', # spatial coordinates
    k = 10, # k nearest neighbors
    n_neighborhoods = 7, # number of CNs (or max number of CNs for elbow plot)
    elbow = False) # if True, will plot the elbow plot

In [ ]:
# to better visualize the cellular neighborhood (CN), we choose a color palette
# but if you set palette = None in the following function, it will randomly generate a palette for you
cn_palette = {
    0: '#7f59b0',
    1: '#63c5ea',
    2: '#ea4b8b',
    3: '#fcb424',
    4: '#575a6d',
    5: '#b4c640',
    6: '#e0e3d2'}

# save the palette in the adata
adata.uns['CN_k10_n7_colors'] = cn_palette.values()

# plot CN to see what cell types are enriched per CN so that we can annotate them better
sp.pl.cn_exp_heatmap(
    adata, # anndata
    cluster_col = "celltypes", # cell type column
    cn_col = "CN_k10_n7", # CN column
    palette=cn_palette, # color palette for CN
    savefig = True, # save the figure
    output_dir = output_dir, # output directory
    output_fname="CN_k10_n7",
    rand_seed = 1, # random seed for reproducibility
    figsize=(10, 6)
)

In [ ]:
# Define neighborhood annotation for every cluster ID
neighborhood_annotation = {
    0: 'Neutrophil Zone',
    1: 'Vascular Zone', 
    2: 'Cytotoxic T-cell Zone',
    3: 'Inflammatory Zone',
    4: 'Follicular Zone', 
    5: 'Immune Priming Zone',
    6: 'Fibrotic Zone'
    }


adata.obs['CN_k10_n7_annot'] = (
    adata.obs['CN_k10_n7']
    .map(neighborhood_annotation)
    .astype('category')
)

In [ ]:
# match the color of the annotated CN to the original CN
cn_annt_palette = {neighborhood_annotation[key]: value for key, value in cn_palette.items()}


# replotting with CN annotation
sp.pl.cn_exp_heatmap(
    adata, 
    cluster_col = "celltypes", 
    cn_col = "CN_k10_n7_annot",
    palette = cn_annt_palette, #if None, there is randomly generated in the code 
    savefig=True,
    output_fname = "CN_k10_n7_annot",
    output_dir = output_dir,
    figsize=(10, 6)
)

In [ ]:
# Convert dict_values to a list
adata.uns['CN_k10_n7_colors'] = list(adata.uns['CN_k10_n7_colors'])

adata.uns['CN_k10_n7_colors'] 

In [ ]:
adata.write(output_dir + '/adata_annotated_cn.h5ad')

In [ ]:
adata = sc.read_h5ad(output_dir + "/adata_annotated_cn.h5ad")

In [ ]:
np.unique(adata.obs['celltypes'])

In [ ]:
output_dir_catplots = root_path + '/Analysis/CN_Corrected_Dormant/Catplots/' # inset your own path
os.makedirs("%s/CN" % output_dir_catplots, exist_ok=True)
os.makedirs("%s/CellType" % output_dir_catplots, exist_ok=True)

In [ ]:
for patient_id in np.unique(adata.obs.code):
    p_adata = adata[adata.obs.code == patient_id]
    cond = np.unique(p_adata.obs['condition_final'])[0]
    sp.pl.catplot(
        p_adata, 
        color = "CN_k10_n7_annot", # specify group column name here (e.g. celltype_fine)
        unique_region = "unique_region", # specify unique_regions here
        X='X', Y='Y', # specify x and y columns here
        n_columns=3, # adjust the number of columns for plotting here (how many plots do you want in one row?)
        palette=cn_annt_palette, #default is None which means the color comes from the anndata.uns that matches the UMAP
        savefig=True, # save figure as pdf
        output_fname = "Catplot_CNs_%s_%s" %(patient_id, cond), # change it to file name you prefer when saving the figure
        output_dir="%s/CN/" % output_dir_catplots, # specify output directory here (if savefig=True)
        figsize= 8, # specify figure size here
        size = 15) # specify size of the points in the plot
    
    sp.pl.catplot(
        p_adata, 
        color = "celltypes", # specify group column name here (e.g. celltype_fine)
        unique_region = "unique_region", # specify unique_regions here
        X='X', Y='Y', # specify x and y columns here
        n_columns=3, # adjust the number of columns for plotting here (how many plots do you want in one row?)
        palette=None, #default is None which means the color comes from the anndata.uns that matches the UMAP
        savefig=True, # save figure as pdf
        output_fname = "Catplot_CellType_%s_%s" %(patient_id, cond), # change it to file name you prefer when saving the figure
        output_dir="%s/CellType/" % output_dir_catplots, # specify output directory here (if savefig=True)
        figsize= 8, # specify figure size here
        size = 15) # specify size of the points in the plot

In [ ]:
np.unique(adata.obs['condition_final'])

In [ ]:
cnmap_dicts = {}

for targeted_group in ['Infected', 'Dormant', 'Uninfected']:
    # We will look at the spatial context maps separately for each condition
    t_adata = adata[adata.obs['condition_final'] == targeted_group]
    #t_adata
    
    cnmap_dicts[targeted_group] = sp.tl.build_cn_map(
        adata = t_adata, # adata object
        cn_col = "CN_k10_n7_annot",# column with CNs
        palette = cn_annt_palette, # color dictionary
        unique_region = 'unique_region',# column with unique regions
        k = 70, # number of neighbors
        X='X', Y='Y',  # coordinates
        threshold = 0.8, # threshold for percentage of cells in CN
        per_keep_thres = 0.8,) # threshold for percentage of cells in CN
    
    cnmap_dicts[targeted_group]['w'].to_csv('%s/CN_MAP_%s.csv' %(output_dir, targeted_group))
    
    # Compute for the frequency of the CNs and paly around with the threshold
    sp.pl.cn_map(cnmap_dict = cnmap_dicts[targeted_group], # dictionary from the previous step
        adata = t_adata, # adata object
        cn_col = "CN_k10_n7_annot", # column with CNs used to color the plot
        palette = cn_annt_palette, # color dictionary
        figsize=(30, 20), # figure size
        savefig=True, # save figure as pdf
        output_fname = "CN_MAP_%s" % targeted_group, # change it to file name you prefer when saving the figure
        output_dir= output_dir # specify output directory here (if savefig=True)
    )

## barycentric projections

In [ ]:
# plot barycentric projections for the tonsil and tonsillitis data

sp.pl.BC_projection(adata=adata[adata.obs['condition_final'] == "Infected"], 
                    output_dir=output_dir, output_fname="/BC_projection1_Infected.png", savefig=True,
    cnmap_dict = cnmap_dicts['Infected'], # dictionary from the previous step
    cn_col = "CN_k10_n7_annot", # column with CNs
    plot_list = ['Inflammatory Zone','Immune Priming Zone', 'Vascular Zone'], # list of CNs to plot (three for the corners)
    cn_col_annt = "CN_k10_n7_annot", # column with CNs used to color the plot
    palette = cn_annt_palette, # color dictionary
    figsize=(6, 6), # figure size
    rand_seed = 1, # random seed for reproducibility
    n_num = None, # number of neighbors
    threshold = 0.6) # threshold for percentage of cells in CN

sp.pl.BC_projection(adata=adata[adata.obs['condition_final'] == "Uninfected"], 
                    output_dir=output_dir, output_fname="/BC_projection1_Uninfected.png", savefig=True,
    cnmap_dict = cnmap_dicts['Uninfected'],
    cn_col = "CN_k10_n7_annot",
    plot_list = ['Inflammatory Zone','Immune Priming Zone', 'Vascular Zone'],
    cn_col_annt = "CN_k10_n7_annot",
    palette = cn_annt_palette,
    figsize=(6, 6),
    rand_seed = 1,
    n_num = None,
    threshold = 0.6)

sp.pl.BC_projection(adata=adata[adata.obs['condition_final'] == "Dormant"], 
                    output_dir=output_dir, output_fname="/BC_projection1_Dormant.png", savefig=True,
    cnmap_dict = cnmap_dicts['Dormant'],
    cn_col = "CN_k10_n7_annot",
    plot_list = ['Inflammatory Zone','Immune Priming Zone', 'Vascular Zone'],
    cn_col_annt = "CN_k10_n7_annot",
    palette = cn_annt_palette,
    figsize=(6, 6),
    rand_seed = 1,
    n_num = None,
    threshold = 0.6)

## Distance Permutation Analysis

t_adata = adata[[(True if cond == 1 or cond == 3 else False) for cond in adata.obs.condition]]

In [ ]:
targeted_col = "celltypes"

In [ ]:
#Identify potential interactions

# compute the potential interactions
distance_pvals, results_dict = sp.tl.identify_interactions(
    adata = adata, # AnnData object
    cellid = "cell_id", # column that contains the cell id (set index if the cell id is the index of the dataframe)
    x_pos = "X", # x coordinate column
    y_pos = "Y", # y coordinate column
    cell_type = targeted_col, # column that contains the cell type information
    region = "unique_region", # column that contains the region information
    num_iterations=1000, # number of iterations for the permutation test
    num_cores=12,  # number of CPU threads to use
    min_observed = 10, # minimum number of observed interactions to consider a cell type pair
    comparison = 'condition_final', # column that contains the condition information we want to compare
    distance_threshold=20/0.5085) # distance threshold in px (20 µm)

In [ ]:
distance_pvals = distance_pvals.rename(columns = {'condition_final': 'condition'})

In [ ]:
distance_pvals_filt = sp.tl.remove_rare_cell_types(adata, 
                       distance_pvals, 
                       cell_type_column=targeted_col, 
                       min_cell_type_percentage=1)

In [ ]:
# Identify significant cell-cell interactions
# dist_table_filt is a simplified table used for plotting
# dist_data_filt contains the filtered raw data with more information about the pairs
#  The function outputs two dataframes:  and dist_data_filt that contains all filtered interactions and  dist_table_filt that contains a table for all interactions that show a significant value in both tissues
dist_table_filt, dist_data_filt = sp.tl.filter_interactions(
    distance_pvals = distance_pvals_filt,
    pvalue = 0.05,
    logfold_group_abs = 0.1,
    comparison = 'condition')

print(dist_table_filt.shape)
#dist_data_filt = dist_data_filt.rename(columns={'condition_final': 'condition'})

dist_data_filt

In [ ]:
sp.pl.plot_top_n_distances(
    dist_table_filt,
    dist_data_filt,
    n=5,
    colors=None,
    dodge=True,
    savefig=True,
    output_fname="Distance_Cells",
    output_dir=output_dir,
    figsize=(5, 5),
    unit="px",
    errorbars=True,
)

In [ ]:
dist_data_filt.condition = dist_data_filt.condition.astype('string')
distance_pvals.condition = distance_pvals.condition.astype('string')

In [ ]:
sp.pl.distance_graph(dist_table = dist_data_filt, # the (filtered) distance data table you want to plot 
                  distance_pvals = distance_pvals, # the full distance data table
                  condition_pair=['Uninfected', 'Dormant'],
                  node_size=1600, font_size=6,
                  palette=None,
                  dpi = 300,
                  savefig=True,
                  output_fname="Dist_Interaction_Uninfected_vs_Dormant",
                  output_dir=output_dir,)

In [ ]:
sp.pl.distance_graph(dist_table = dist_data_filt, # the (filtered) distance data table you want to plot 
                  distance_pvals = distance_pvals, # the full distance data table
                  condition_pair=['Infected', 'Dormant'],
                  node_size=1600, font_size=6,
                  palette=None,
                  dpi = 300,
                  savefig=True,
                  output_fname="Dist_Interaction_Infected_vs_Dormant",
                  output_dir=output_dir,)

In [ ]:
sp.pl.distance_graph(dist_table = dist_data_filt, # the (filtered) distance data table you want to plot 
                  distance_pvals = distance_pvals, # the full distance data table
                  condition_pair=['Uninfected', 'Infected'],
                  node_size=1600, font_size=6,
                  palette=None,
                  dpi = 300,
                  savefig=True,
                  output_fname="Dist_Interaction_Uninfected_vs_Infected",
                  output_dir=output_dir,)

In [ ]:
adata.obs.to_excel("%s/Metadata.xlsx" % output_dir)

###  Patch Proximity Analysis

In [ ]:
np.unique(adata.obs['group'])

In [ ]:
n_targeted_group = "Infected"
#n_targeted_group = "Dormant"
#n_targeted_group = "Uninfected"

t_adata = adata[adata.obs['group'] == n_targeted_group]

t_adata

In [ ]:
np.unique(t_adata.obs.unique_region)

## Compute for proximal cells
### unique_region - fov
### CN_k10_n8_annot - name of CNs

In [ ]:
adata.obs.columns

In [ ]:
np.unique(adata.obs['CN_k10_n7_annot'])

In [ ]:
targeted_CN = "Immune Priming Zone"
targeted_CN = "Cytotoxic T-cell Zone"
targeted_CN = "Inflammatory Zone"
targeted_CN = "Vascular Zone"
targeted_CN = "Fibrotic Zone"

key_name = 'ppa_result_20_40_60_80_100_border_cell_radius_%s' % targeted_CN.replace(" ", "")

print(key_name)

In [ ]:
if key_name not in adata.uns.keys():
    # this region result is also saved to adata.uns
    results, outlines_results = sp.tl.patch_proximity_analysis(
        adata, # the annotated adata object
        region_column = "unique_region", # column with the region information
        patch_column = "CN_k10_n7_annot", # column with the patch information (derive patches from this column)
        group=targeted_CN, # group to consider
        min_cluster_size=50, # minimum cluster size to consider
        x_column='X', y_column='Y', # spatial coordinates
        radius = [20, 40, 60, 80, 100], # to get the distance in µm
        edge_neighbours = 1, # number of neighbours to consider for edge detection if set to 1 only the hull is considered
        plot = True, # plot the results for demonstration and/or documentation (set to False to skip plotting - improves speed)
        original_unit_scale = 0.377, # scale factor for the units (1 = 1px per unit e.g. µm)
        method= "border_cell_radius", # method to use for the edge detection
        key_name = key_name, # key name to store the result in adata.uns
        save_geojson = False, # save the results as geojson
        ) # plot detection for demonstration purposes

    adata.uns[key_name].to_excel("%s/PatchProximity_%s.xlsx" %(output_dir, targeted_CN.replace(" ", "")))


In [ ]:
color_map = {
    'B Inactive': '#4682B4',
    'B Suppressive': '#B0C4DE',
    'DCs Active': '#FF8C00',
    'Endo': '#FFDAB9',
    'Fibroblast': '#228B22',
    'M1-like Active': '#90EE90',
    'M1-like Inactive': '#B22222',
    'M2_TAM Active': '#F08080',
    'M2_TAM Inactive': '#9370DB',
    'M2_TAM Suppressive': '#D8BFD8',
    'Mast': '#8B4513',
    'Neutrophil': '#BC8F8F',
    'T (pan)': '#DA70D6',
    'T CD4 Mem': '#FFB6C1',
    'T CD4 NK-like': '#808080',
    'T CD8 Cytotoxic': '#BDB76B',
    'T CD8 Naive': '#E6E6FA',
    'T CD8 Active': '#00CED1'
}
#plot_colors = [color_map[ct] for ct in labels]

In [ ]:
targeted_CNs = ["Immune Priming Zone", "Cytotoxic T-cell Zone", "Inflammatory Zone", 
                "Vascular Zone", "Fibrotic Zone"]


for targeted_CN in targeted_CNs:
    key_name = 'ppa_result_20_40_60_80_100_border_cell_radius_%s' % targeted_CN.replace(" ", "")

    print(key_name)

    re_distance = np.array(["<20µm" if d < 20 else "<40µm" for d in adata.uns[key_name]['distance_from_patch']])
    re_distance


    t_df = pd.DataFrame(np.transpose([re_distance, 
                                      adata.uns[key_name]['celltypes'].values, 
                                      adata.uns[key_name]['condition_final'].values]), 
                        columns=["Distance", "Celltypes", "Group"])
    print(t_df)

    # 1. Calculate Raw Counts per Group and Distance
    counts = t_df.groupby(['Group', 'Distance', 'Celltypes']).size().reset_index(name='Count')

    # 2. Compute Cumulative Counts
    # We create a new dataframe to hold the cumulative totals
    cumulative_list = []

    for group in counts['Group'].unique():
        group_data = counts[counts['Group'] == group]

        # Zone 1: <20μm (just the inner cells)
        inner_zone = group_data[group_data['Distance'] == '<20µm'].copy()
        inner_zone['Cumulative_Count'] = inner_zone['Count']
        cumulative_list.append(inner_zone)

        # Zone 2: >=20μm (Inner cells + Outer cells)
        # This represents the "Total" area
        total_area = group_data.groupby('Celltypes')['Count'].sum().reset_index()
        total_area['Distance'] = '<40µm'
        total_area['Group'] = group
        total_area['Cumulative_Count'] = total_area['Count']
        cumulative_list.append(total_area)

    df_cumulative = pd.concat(cumulative_list)


    # 3. Calculate Proportions based on Cumulative Totals
    df_cumulative['Total_Area_Count'] = df_cumulative.groupby(['Distance', 'Group'])['Cumulative_Count'].transform('sum')
    df_cumulative['Proportion'] = df_cumulative['Cumulative_Count'] / df_cumulative['Total_Area_Count']

    df_cumulative.to_excel("%s/Prop_%s.xlsx" %(output_dir, targeted_CN))
    
    
    # We'll filter for a specific Group (e.g., 'Uninfected') for this plot
    # If you have multiple groups, you would loop this or create subplots

    for group_label in ['Uninfected', 'Dormant', 'Infected']:
        plot_df = df_cumulative[df_cumulative['Group'] == group_label]

        # Pivot so we can easily access each distance ring
        pivot_df = plot_df.pivot(index='Celltypes', columns='Distance', values='Proportion').fillna(0)
        pivot_df = pivot_df.reindex(color_map.keys(), fill_value=0)
        plot_colors = [color_map[ct] for ct in pivot_df.index]
        #pivot_df
        #plot_colors

        ring_inner_data = pivot_df.iloc[:, 0].values
        ring_outer_data = pivot_df.iloc[:, 1].values
        labels = pivot_df.index.tolist()

        # 3. Generate the plot
        fig, ax = plt.subplots(figsize=(16, 8))
        ring_width = 0.2
        # Outer Ring (>=20)
        ax.pie(ring_outer_data, radius=1.0, colors=plot_colors,
               startangle=90, counterclock=False,
               wedgeprops=dict(width=ring_width, edgecolor='white', linewidth=0.5))

        # Inner Ring (<20)
        ax.pie(ring_inner_data, radius=0.8, colors=plot_colors,
               startangle=90, counterclock=False,
               wedgeprops=dict(width=ring_width, edgecolor='white', linewidth=0.5))

        # 4. Show Titles for each Ring
        # We place these at the top center (x=0) within the ring's width
        # Ring labels: 0.9 is the middle of the outer ring, 0.7 is the middle of the inner ring
        ax.text(0, 0.9, 'Total (<40µm)', ha='center', va='center', weight='bold', fontsize=10)
        ax.text(0, 0.7, '<20µm', ha='center', va='center', weight='bold', fontsize=10)

        # Styling
        ax.text(0, 1.25, 'Cell types around %s - %s' %(targeted_CN, group_label), ha='center', va='center', 
                fontsize=12, fontweight='bold')
        ax.legend(labels, loc="center left", bbox_to_anchor=(1.1, 0.5), frameon=False)
        ax.set(aspect="equal")

        fig.savefig("%s/Donut_Merged_%s_%s.png" %(output_dir, group_label, targeted_CN))

        plt.close(fig)

#df_cumulative

### Donut

In [ ]:
targeted_CN = "Immune Priming Zone"
targeted_CN = "Cytotoxic T-cell Zone"
targeted_CN = "Inflammatory Zone"
targeted_CN = "Vascular Zone"
targeted_CN = "Fibrotic Zone"

key_name = 'ppa_result_20_40_60_80_100_border_cell_radius_%s' % targeted_CN.replace(" ", "")

print(key_name)

In [ ]:
sp.pl.ppa_res_donut(
    adata,
    cat_col = 'celltypes',
    key_name=key_name,
    palette=None,
    distance_mode="within",  # "within" or "between"
    unit="µm",
    figsize=(10, 10),
    add_guides=True,
    text="Cell types around %s" % targeted_CN,
    label_color="black",
    group_by= 'condition_final',
    title="PPA (%s - within)" % targeted_CN, 
    output_dir = output_dir,
    output_fname = "Donut_plot_PPA_within_celltypes_%s" %(targeted_CN.replace(" ", "")),
    savefig=True
)

In [ ]:
sp.pl.ppa_res_donut(
    adata,
    cat_col = 'celltypes',
    key_name=key_name,
    palette=None,
    distance_mode="between",  # "within" or "between"
    unit="µm",
    figsize=(10, 10),
    add_guides=True,
    text="Cell types around %s" % targeted_CN,
    label_color="black",
    group_by= 'condition_final',
    title="PPA (%s - between)" % targeted_CN, 
    output_dir = output_dir,
    output_fname = "Donut_plot_PPA_between_celltypes_%s" %(targeted_CN.replace(" ", "")),
    savefig=True
)

In [ ]:
adata.write(output_dir + '/adata_annotated_cn.h5ad')

In [ ]:
adata.obs.to_excel(output_dir + "/Metadata.xlsx")

In [ ]:
adata = sc.read_h5ad(output_dir + '/adata_annotated_cn.h5ad')